In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [4]:
def fetch(url):

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (research data collection)"
        },
        timeout=15
    )

    response.raise_for_status()

    return response.text

In [5]:
url = "https://www.setopati.com/art"

html = fetch(url)

print(len(html))

80958


In [6]:
soup = BeautifulSoup(
    html,
    "lxml"
)

print(soup.title.get_text(strip=True))

Nepal's Digital Newspaper :: नेपालको डिजिटल पत्रिका :: Setopati


In [7]:
for a in soup.find_all("a", href=True):

    title = a.get_text(
        " ",
        strip=True
    )

    url = urljoin(
        "https://www.setopati.com",
        a["href"]
    )

    if title:

        print(title)
        print(url)
        print("-" * 50)

YouTube
https://www.youtube.com/user/setopati
--------------------------------------------------
Subscribe
https://www.setopati.com/subscribe
--------------------------------------------------
युनिकोड
https://www.setopati.com/unicode-converter
--------------------------------------------------
EN English
https://en.setopati.com/
--------------------------------------------------
गृहपृष्ठ
https://www.setopati.com
--------------------------------------------------
कभर स्टोरी
https://www.setopati.com/cover-story
--------------------------------------------------
राजनीति
https://www.setopati.com/politics
--------------------------------------------------
बजार अर्थतन्त्र
https://www.setopati.com/kinmel
--------------------------------------------------
नेपाली ब्रान्ड
https://www.setopati.com/nepali-brand
--------------------------------------------------
विचार
https://www.setopati.com/opinion
--------------------------------------------------
समाज
https://www.setopati.com/social
-----------

In [8]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

url = "https://www.setopati.com/art"

response = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0 (research data collection)"
    },
    timeout=15
)

response.raise_for_status()

html = response.text

soup = BeautifulSoup(
    html,
    "lxml"
)


article_links = []


for a in soup.find_all("a", href=True):

    title = a.get_text(
        " ",
        strip=True
    )

    article_url = urljoin(
        url,
        a["href"]
    )

    # Check for Setopati entertainment article pattern
    if re.match(
        r"https://www\.setopati\.com/art/[^/]+/\d+",
        article_url
    ):

        if title:

            article_links.append({
                "title": title,
                "url": article_url
            })


# Remove duplicates
unique_links = {}

for article in article_links:

    unique_links[
        article["url"]
    ] = article["title"]


print(
    "Articles found:",
    len(unique_links)
)


for i, (url, title) in enumerate(
    unique_links.items(),
    start=1
):

    print(
        f"{i}. {title}"
    )

    print(
        url
    )

    print(
        "-" * 50
    )

Articles found: 30
1. कला समाचार 'साम्राज्य'मा सौगातपछि दयाहाङ अनुबन्धित
https://www.setopati.com/art/art-activity/395736
--------------------------------------------------
2. बलिउड/हलिउड चर्चित सिरिज 'मिर्जापुर' बन्यो फिल्म, यस्तो छ 'मिर्जापुरः द मुभी'को ट्रेलर
https://www.setopati.com/art/bollywood/395719
--------------------------------------------------
3. बलिउड/हलिउड पछि सर्‍यो श्रद्धा कपूरको 'ईथा', टर्‍यो 'टक्सिक'सँगको भिड्न्त
https://www.setopati.com/art/bollywood/395712
--------------------------------------------------
4. कला समाचार एप्पल इन्टरटेन्मेन्टले फिल्म 'भिडियोटेप' बनाउने, शीर्ष भूमिकामा उपासना-कुन्दन
https://www.setopati.com/art/art-activity/395670
--------------------------------------------------
5. कला समाचार प्रियंकाको 'मास्टर्नी' मा चर्चित गीत 'गुराँस फुल्दा' को नयाँ भर्सन (भिडिओ)
https://www.setopati.com/art/art-activity/395636
--------------------------------------------------
6. कला समाचार नीता ढुंगाना निर्देशित 'पार्वती'को गीत 'मेरी दिदी' सार्वजनिक
https://ww

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import json
import time
import random
import os


# ============================================================
# SETTINGS
# ============================================================

BASE_URL = "https://www.setopati.com"

START_URL = "https://www.setopati.com/art"

TARGET_ARTICLES = 200

OUTPUT_FILE = "setopati_entertainment_200.json"


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (research data collection)"
})


# ============================================================
# FETCH PAGE
# ============================================================

def fetch(url, max_retries=3):

    for attempt in range(max_retries):

        try:

            print(f"\nFetching: {url}")

            response = session.get(
                url,
                timeout=15
            )

            response.raise_for_status()

            # Be polite to the website
            time.sleep(
                random.uniform(1.5, 3.0)
            )

            return response.text

        except requests.RequestException as e:

            print(
                f"[Retry {attempt + 1}] {e}"
            )

            time.sleep(
                2 ** attempt
            )


    print(f"[FAILED] {url}")

    return None


# ============================================================
# EXTRACT ARTICLE LINKS FROM /art PAGE
# ============================================================

def get_article_links(html):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    links = {}

    for a in soup.find_all(
        "a",
        href=True
    ):

        title = a.get_text(
            " ",
            strip=True
        )

        article_url = urljoin(
            BASE_URL,
            a["href"]
        )

        # ----------------------------------------------------
        # Setopati entertainment article pattern:
        #
        # /art/art-activity/395736
        # /art/bollywood/395719
        # /art/music/395560
        # /art/book/395512
        #
        # ----------------------------------------------------

        if re.match(
            r"https://www\.setopati\.com/art/[^/]+/\d+/?$",
            article_url
        ):

            if title:

                links[article_url] = title


    return links


# ============================================================
# COLLECT ARTICLES FROM MULTIPLE PAGES
# ============================================================

def collect_article_links(target):

    all_articles = {}

    page = 1


    while len(all_articles) < target:

        page_url = START_URL

        if page > 1:

            page_url = (
                f"{START_URL}?page={page}"
            )


        print("\n" + "=" * 60)

        print(
            f"LISTING PAGE {page}"
        )

        print(
            f"Articles collected: "
            f"{len(all_articles)}"
        )

        print("=" * 60)


        html = fetch(
            page_url
        )

        if html is None:

            break


        page_articles = get_article_links(
            html
        )


        print(
            f"Articles found on page: "
            f"{len(page_articles)}"
        )


        # ----------------------------------------------------
        # Add to global collection
        # ----------------------------------------------------

        for url, title in page_articles.items():

            if url not in all_articles:

                all_articles[url] = title


        print(
            f"Total unique articles: "
            f"{len(all_articles)}"
        )


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if len(page_articles) == 0:

            print(
                "No articles found. "
                "Stopping pagination."
            )

            break


        page += 1


    return all_articles


# ============================================================
# CLEAN NEPALI TEXT
# ============================================================

def clean_nepali_text(text):

    # Keep Devanagari + common Nepali punctuation
    text = re.sub(
        r'[^\u0900-\u097F\s।॥,!?():\-\'"“”‘’]',
        ' ',
        text
    )

    # Remove excessive whitespace
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()


# ============================================================
# EXTRACT ARTICLE TEXT
# ============================================================

def extract_article(article_url):

    html = fetch(
        article_url
    )

    if html is None:

        return None, None


    soup = BeautifulSoup(
        html,
        "lxml"
    )


    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title_tag = soup.find("h1")

    if title_tag:

        title = title_tag.get_text(
            " ",
            strip=True
        )

    else:

        title = ""


    # --------------------------------------------------------
    # ARTICLE PARAGRAPHS
    # --------------------------------------------------------

    paragraphs = []


    for p in soup.find_all("p"):

        text = p.get_text(
            " ",
            strip=True
        )


        if not text:

            continue


        # Keep only paragraphs containing Nepali
        if re.search(
            r"[\u0900-\u097F]",
            text
        ):

            cleaned = clean_nepali_text(
                text
            )


            if cleaned:

                paragraphs.append(
                    cleaned
                )


    # --------------------------------------------------------
    # Combine paragraphs
    # --------------------------------------------------------

    article_text = "\n".join(
        paragraphs
    )


    return title, article_text


# ============================================================
# SAVE JSON
# ============================================================

def save_json(data):

    with open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=4
        )


# ============================================================
# MAIN
# ============================================================

print(
    "\nCollecting article URLs..."
)


all_articles = collect_article_links(
    TARGET_ARTICLES
)


print("\n" + "=" * 60)

print(
    f"TOTAL ARTICLES AVAILABLE: "
    f"{len(all_articles)}"
)

print("=" * 60)


# ============================================================
# RANDOMLY SELECT 75 ARTICLES
# ============================================================

article_items = list(
    all_articles.items()
)


if len(article_items) < TARGET_ARTICLES:

    print(
        f"\nOnly {len(article_items)} "
        f"articles were found."
    )

    selected_articles = article_items

else:

    selected_articles = random.sample(
        article_items,
        TARGET_ARTICLES
    )


print(
    f"\nRandomly selected "
    f"{len(selected_articles)} articles."
)


# ============================================================
# JSON STRUCTURE
# ============================================================

dataset = {
    "articles": []
}


# ============================================================
# SCRAPE ARTICLES
# ============================================================

for index, (url, listing_title) in enumerate(
    selected_articles,
    start=1
):


    print("\n" + "-" * 60)

    print(
        f"ARTICLE {index}/{len(selected_articles)}"
    )

    print(url)

    print("-" * 60)


    title, text = extract_article(
        url
    )


    # --------------------------------------------------------
    # Skip failed articles
    # --------------------------------------------------------

    if not text or len(text) < 100:

        print(
            "[SKIPPED] Article text too short"
        )

        continue


    # --------------------------------------------------------
    # Create record
    # --------------------------------------------------------

    record = {

        "id": len(dataset["articles"]) + 1,

        "url": url,

        "title": title
        if title
        else listing_title,

        "language": "ne",

        "text": text
    }


    # --------------------------------------------------------
    # Add to dataset
    # --------------------------------------------------------

    dataset["articles"].append(
        record
    )


    # --------------------------------------------------------
    # SAVE IMMEDIATELY
    #
    # This is important.
    # If the program stops at article 50,
    # articles 1-49 are already saved.
    # --------------------------------------------------------

    save_json(
        dataset
    )


    print(
        f"[SAVED] Article "
        f"{len(dataset['articles'])}"
    )

    print(
        f"Characters: {len(text)}"
    )


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 60)

print(
    "SCRAPING COMPLETE"
)

print(
    f"Articles successfully saved: "
    f"{len(dataset['articles'])}"
)

print(
    f"JSON file: {OUTPUT_FILE}"
)

print("=" * 60)



LISTING PAGE 1
Articles collected: 0

Fetching: https://www.setopati.com/art
Articles found on page: 30
Total unique articles: 30

LISTING PAGE 2
Articles collected: 30

Fetching: https://www.setopati.com/art?page=2
Articles found on page: 30
Total unique articles: 60

LISTING PAGE 3
Articles collected: 60

Fetching: https://www.setopati.com/art?page=3
Articles found on page: 30
Total unique articles: 90

LISTING PAGE 4
Articles collected: 90

Fetching: https://www.setopati.com/art?page=4
Articles found on page: 30
Total unique articles: 120

LISTING PAGE 5
Articles collected: 120

Fetching: https://www.setopati.com/art?page=5
Articles found on page: 30
Total unique articles: 150

LISTING PAGE 6
Articles collected: 150

Fetching: https://www.setopati.com/art?page=6
Articles found on page: 30
Total unique articles: 180

LISTING PAGE 7
Articles collected: 180

Fetching: https://www.setopati.com/art?page=7
Articles found on page: 30
Total unique articles: 210

TOTAL ARTICLES AVAILABLE: 

In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import json
import time
import random


# ============================================================
# SETTINGS
# ============================================================

BASE_URL = "https://www.setopati.com"

START_URL = "https://www.setopati.com/global"

TARGET_ARTICLES = 200

OUTPUT_FILE = "setopati_global_75.json"


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (research data collection)"
})


# ============================================================
# FETCH
# ============================================================

def fetch(url, max_retries=3):

    for attempt in range(max_retries):

        try:

            print(f"\nFetching: {url}")

            response = session.get(
                url,
                timeout=15
            )

            response.raise_for_status()

            time.sleep(
                random.uniform(1.5, 3.0)
            )

            return response.text

        except requests.RequestException as e:

            print(
                f"[Retry {attempt + 1}] {e}"
            )

            time.sleep(
                2 ** attempt
            )

    print(f"[FAILED] {url}")

    return None


# ============================================================
# FIND GLOBAL ARTICLE LINKS
# ============================================================

def get_article_links(html):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    links = {}

    for a in soup.find_all(
        "a",
        href=True
    ):

        title = a.get_text(
            " ",
            strip=True
        )

        article_url = urljoin(
            BASE_URL,
            a["href"]
        )

        # ----------------------------------------------------
        # GLOBAL ARTICLE URL
        #
        # Examples:
        #
        # /global/america/395xxx
        # /global/europe/395xxx
        # /global/australia/395xxx
        # /global/other/395xxx
        #
        # ----------------------------------------------------

        if re.match(
            r"https://www\.setopati\.com/global/[^/]+/\d+/?$",
            article_url
        ):

            if title:

                links[article_url] = title

    return links


# ============================================================
# COLLECT ARTICLE LINKS FROM MULTIPLE PAGES
# ============================================================

def collect_article_links(target):

    all_articles = {}

    page = 1

    while len(all_articles) < target:

        if page == 1:

            page_url = START_URL

        else:

            page_url = (
                f"{START_URL}?page={page}"
            )


        print("\n" + "=" * 60)

        print(
            f"LISTING PAGE: {page}"
        )

        print(
            f"ARTICLES COLLECTED: "
            f"{len(all_articles)}"
        )

        print("=" * 60)


        html = fetch(
            page_url
        )

        if html is None:

            break


        page_articles = get_article_links(
            html
        )


        print(
            f"Articles found on page: "
            f"{len(page_articles)}"
        )


        # Add new articles
        for url, title in page_articles.items():

            if url not in all_articles:

                all_articles[url] = title


        print(
            f"Total unique articles: "
            f"{len(all_articles)}"
        )


        # No articles means stop
        if not page_articles:

            print(
                "No more articles found."
            )

            break


        page += 1


    return all_articles


# ============================================================
# CLEAN NEPALI TEXT
# ============================================================

def clean_nepali_text(text):

    text = re.sub(
        r'[^\u0900-\u097F\s।॥,!?():\-\'"“”‘’]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()


# ============================================================
# EXTRACT ARTICLE
# ============================================================

def extract_article(url):

    html = fetch(url)

    if html is None:

        return None, None


    soup = BeautifulSoup(
        html,
        "lxml"
    )


    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title_tag = soup.find("h1")

    if title_tag:

        title = title_tag.get_text(
            " ",
            strip=True
        )

    else:

        title = ""


    # --------------------------------------------------------
    # PARAGRAPHS
    # --------------------------------------------------------

    paragraphs = []


    for p in soup.find_all("p"):

        text = p.get_text(
            " ",
            strip=True
        )


        if not text:

            continue


        # Check for Devanagari/Nepali
        if re.search(
            r"[\u0900-\u097F]",
            text
        ):

            cleaned = clean_nepali_text(
                text
            )

            if cleaned:

                paragraphs.append(
                    cleaned
                )


    # --------------------------------------------------------
    # COMBINE
    # --------------------------------------------------------

    article_text = "\n".join(
        paragraphs
    )


    return title, article_text


# ============================================================
# SAVE JSON
# ============================================================

def save_json(dataset):

    with open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dataset,
            f,
            ensure_ascii=False,
            indent=4
        )


# ============================================================
# MAIN
# ============================================================

print(
    "Collecting Global article URLs..."
)


all_articles = collect_article_links(
    TARGET_ARTICLES
)


print("\n" + "=" * 60)

print(
    f"TOTAL ARTICLES FOUND: "
    f"{len(all_articles)}"
)

print("=" * 60)


# ============================================================
# RANDOMLY SELECT 75
# ============================================================

article_items = list(
    all_articles.items()
)


if len(article_items) <= TARGET_ARTICLES:

    selected_articles = article_items

else:

    selected_articles = random.sample(
        article_items,
        TARGET_ARTICLES
    )


print(
    f"\nSelected "
    f"{len(selected_articles)} articles."
)


# ============================================================
# JSON STRUCTURE
# ============================================================

dataset = {
    "articles": []
}


# ============================================================
# SCRAPE EACH ARTICLE
# ============================================================

for index, (url, listing_title) in enumerate(
    selected_articles,
    start=1
):

    print("\n" + "-" * 60)

    print(
        f"ARTICLE {index}/{len(selected_articles)}"
    )

    print(url)

    print("-" * 60)


    title, text = extract_article(
        url
    )


    # --------------------------------------------------------
    # Skip failed/short articles
    # --------------------------------------------------------

    if not text or len(text) < 100:

        print(
            "[SKIPPED] Text too short"
        )

        continue


    # --------------------------------------------------------
    # CREATE RECORD
    # --------------------------------------------------------

    record = {

        "id": len(dataset["articles"]) + 1,

        "url": url,

        "title": (
            title
            if title
            else listing_title
        ),

        "language": "ne",

        "text": text
    }


    # --------------------------------------------------------
    # ADD TO JSON
    # --------------------------------------------------------

    dataset["articles"].append(
        record
    )


    # --------------------------------------------------------
    # SAVE IMMEDIATELY
    # --------------------------------------------------------

    save_json(
        dataset
    )


    print(
        f"[SAVED] Article "
        f"{len(dataset['articles'])}"
    )

    print(
        f"Characters: {len(text)}"
    )


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 60)

print("SCRAPING COMPLETE")

print(
    f"Successfully saved: "
    f"{len(dataset['articles'])} articles"
)

print(
    f"Output file: {OUTPUT_FILE}"
)

print("=" * 60)


LISTING PAGE: 1
ARTICLES COLLECTED: 0

Fetching: https://www.setopati.com/global
Articles found on page: 30
Total unique articles: 30

LISTING PAGE: 2
ARTICLES COLLECTED: 30

Fetching: https://www.setopati.com/global?page=2
Articles found on page: 30
Total unique articles: 60

LISTING PAGE: 3
ARTICLES COLLECTED: 60

Fetching: https://www.setopati.com/global?page=3
Articles found on page: 30
Total unique articles: 90

TOTAL ARTICLES FOUND: 90

Selected 75 articles.

------------------------------------------------------------
ARTICLE 1/75
https://www.setopati.com/global/other-global/395603
------------------------------------------------------------

Fetching: https://www.setopati.com/global/other-global/395603
[SAVED] Article 1
Characters: 2422

------------------------------------------------------------
ARTICLE 2/75
https://www.setopati.com/global/other-global/393550
------------------------------------------------------------

Fetching: https://www.setopati.com/global/other-global/